In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import binned_statistic
import seaborn as sns

In [ ]:
# To create a new result container 
Residuals_df = pd.DataFrame()
# To load an existing one
#Residuals_df = pd.read_csv(r'C:\Users\liliv\Desktop\Brady Hot Springs\sensitivity_gid\sensitivity\residual_error.csv')

# Load data 

In [ ]:
### INFO ###
path_model = r'C:\Users\liliv\Desktop\Brady Hot Springs\DAS_data'
path_measure = r'C:\Users\liliv\Desktop\Brady Hot Springs\DAS_data'
model_type = 'T'
sweep_number = 5491
amplitude_file_name = 9
smoothing_channel = 3
SNR = 2

In [ ]:
model_df = pd.read_csv(path_model+'\{}.csv'.format(amplitude_file_name))

model_amplitude = np.convolve(model_amplitude, np.ones(smoothing_channel)/smoothing_channel, mode='same')

DAS_measure_df = pd.read_csv(path_measure+'\Site{}\RMS_0_3.csv'.format(site))
DAS_amplitude = DAS_measure_df['RMS_'+str(sweep_number)]
DAS_amplitude = 11.6*DAS_amplitude*1e-9

DAS_amplitude = np.convolve(DAS_amplitude, np.ones(smoothing_channel)/smoothing_channel, mode='same')
DAS_noise = DAS_measure_df['noise_'+str(sweep_number)]
DAS_noise = 11.6*DAS_noise*1e-9
DAS_noise = np.convolve(DAS_noise, np.ones(smoothing_channel)/smoothing_channel, mode='same')
channels = DAS_measure_df['Unnamed: 0']

### Remove bad channel (low SNR) ###
SNR_masc = np.where(DAS_amplitude / DAS_noise >=SNR, True, False)


## RUN linear regression

In [ ]:

# Apply seaborn style
sns.set(style="whitegrid")

# --- CLEAN DATA ---
valid_mask = ~np.isnan(model_amplitude) & ~np.isnan(DAS_amplitude) & SNR_masc
measurement = DAS_amplitude[valid_mask]*10e9 # convert to nano strain
model = model_amplitude[valid_mask]
measurement_error = model_error[valid_mask]

# Convert to log
measurement_log = np.log10(measurement)
model_log = np.log10(model)
measurement_error_log = measurement_error / (model * np.log(10))

# --- BINNING ---
n_bins = 200
bin_stat_mean_meas, bin_edges, _ = binned_statistic(measurement_log, measurement_log, statistic='median', bins=n_bins)
bin_stat_mean_model = binned_statistic(measurement_log, model_log, statistic='median', bins=n_bins)[0]
bin_stat_error = binned_statistic(measurement_log, measurement_error_log, statistic='median', bins=n_bins)[0]

# Remove NaNs after binning
valid_bins = ~np.isnan(bin_stat_mean_meas) & ~np.isnan(bin_stat_mean_model) & ~np.isnan(bin_stat_error)
X_binned = sm.add_constant(bin_stat_mean_meas[valid_bins])
y_binned = bin_stat_mean_model[valid_bins]
weights_binned = 1 / (bin_stat_error[valid_bins] ** 2)

# --- FIT WLS ON BINNED DATA ---
model_binned = sm.WLS(y_binned, X_binned, weights=weights_binned)
results_binned = model_binned.fit()
y_pred_binned = results_binned.predict(X_binned)

# Confidence interval for regression
pred_summary = results_binned.get_prediction(X_binned).summary_frame(alpha=0.05)
ci_lower_binned = pred_summary["mean_ci_lower"]
ci_upper_binned = pred_summary["mean_ci_upper"]




# ---- STEP 3: Use params to get predictions for UNBINNED data ----
X_unbinned = sm.add_constant(measurement_log)
y_pred_unbinned = results_binned.predict(X_unbinned)

# ---- STEP 4: Compute unbinned residuals ----
residuals = model_log - y_pred_unbinned


# --- PLOTTING ---
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.05)

# Axes
ax_main = fig.add_subplot(gs[0])
ax_bottom = fig.add_subplot(gs[1], sharex=ax_main)

# --- MAIN PLOT ---
# All data (transparent)
ax_main.errorbar(
    measurement_log,
    model_log,
    yerr=measurement_error_log,
    fmt='o',
    alpha=0.2,
    label='Raw Data ± error',
    color='blue',
    markersize=3,
    capsize=2
)

# Binned points
ax_main.errorbar(
    bin_stat_mean_meas[valid_bins],
    bin_stat_mean_model[valid_bins],
    yerr=bin_stat_error[valid_bins],
    fmt='o',
    color='black',
    ecolor='black',
    elinewidth=1,
    capsize=3,
    markersize=5,
    label='Binned median ± error'
)

# Regression line
ax_main.plot(
    bin_stat_mean_meas[valid_bins],
    y_pred_binned,
    color='#1f77b4',
    linewidth=2.5,
    label='Linear model'
)

'''# Confidence interval
ax_main.fill_between(
    bin_stat_mean_meas[valid_bins],
    ci_lower_binned,
    ci_upper_binned,
    color='#1f77b4',
    alpha=0.2,
    label='95% Confidence Interval'
)'''

# Decorations
ax_main.set_ylabel("Model amplitude (log$_{10}$)", fontsize=12)
ax_main.set_title("Linear projection of the modeled and measured amplitude in log-log space", fontsize=14)
ax_main.legend(fontsize=10)
ax_main.text(
    0.5, 0.93,
    f"$R^2$ = {results_binned.rsquared:.3f}",
    transform=ax_main.transAxes,
    fontsize=12,
    bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray')
)
ax_main.set_xlabel(None)
# --- HISTOGRAM ---
ax_bottom.hist(measurement_log, bins=100, color='darkgray', alpha=0.7, label='Raw measurements')
ax_bottom.hist(bin_stat_mean_meas[valid_bins], bins=100, color='tomato', alpha=0.5, label='Binned medians')
ax_bottom.set_ylabel("Frequency", fontsize=11)
ax_bottom.set_xlabel("Measured RMS (log$_{10}$(n$\epsilon$/s))", fontsize=12)
ax_bottom.legend(loc='upper left', fontsize=9)
ax_bottom.set_yticks([])

# Layout adjustments
for ax in [ax_main, ax_bottom]:
    ax.tick_params(labelsize=10)

#sns.despine(trim=True)
#plt.tight_layout()
plt.show()


## Plot amplitude model and residual

Re-create the fiber record with nan for channels with SNR < 2

In [ ]:
position = 0
container_residuals = np.array([])
for value_quality in valid_mask:
    if value_quality:
        container_residuals = np.append(container_residuals, -residuals[position])
        position += 1

    else: 
        container_residuals = np.append(container_residuals, np.nan)


In [ ]:
# Create subplots with shared x-axis
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 6), gridspec_kw={'height_ratios': [3, 1]})

# ==== Plot 1: Amplitudes ====
ax1.plot(channels, model_amplitude, label="Model",lw = 1, color="tab:blue")
ax1.set_title("Comparaison between RMS amplitude and modeled amplitude (sweep n°{})".format(sweep_number), fontsize=14)
ax1.set_yscale('log')

# Twin axis to fill under measured amplitude
ax1_twin = ax1.twinx()
ax1_twin.set_yscale('log')
ax1_twin.plot(channels, DAS_amplitude*10e9, label="Measured",lw = 0.5, color="black", alpha=0.8)
ax1_twin.fill_between(channels, 0, DAS_amplitude*10e9, where=SNR_masc, color='green', alpha=0.3, label='SNR >=2')
ax1_twin.fill_between(channels, 0, DAS_amplitude*10e9, where=~SNR_masc, color='red', alpha=0.3, label='SNR <2')
#ax1_twin.set_yticks([])  # hide y-axis ticks

# Legends
ax1.legend(loc="upper right")
ax1.set_ylabel("Model Amplitude (log$_{10}$)")
ax1_twin.set_ylabel("RMS Amplitude (log$_{10}$(n$\epsilon$/s))")
ax1_twin.legend(loc="upper left")

# ==== Plot 2: Residuals with error ====
ax2.plot(channels, container_residuals, label="Residuals", color='black')
ax2.fill_between(channels, container_residuals - model_error, container_residuals + model_error,
                 color='gray', alpha=0.4, label="± Error")

ax2.axhline(0, color='red', linestyle='--', linewidth=1)
ax2.set_xlabel("Optical Distance (m)")
ax2.set_ylabel("Log Residual")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

Save results in residual container

In [ ]:
Residuals_df['residual_'+str(sweep_number)] = container_residuals
Residuals_df['error_'+str(sweep_number)] = model_error
Residuals_df['r2_'+str(sweep_number)] =results_binned.rsquared


## Save

In [ ]:
Residuals_df.index = channels
Residuals_df.to_csv(r'C:\Users\liliv\Desktop\Brady Hot Springs\sensitivity_gid\sensitivity\residual_error_site{}.csv'.format(site))